In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import os
import urllib.request
import zipfile
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [6]:
print("Current working directory:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

Current working directory:
C:\Users\UserH\Desktop\NLP\NLP_New\Deep Learning for NLP_Day5

Files:
['assignment1.ipynb', 'assignment2.ipynb', 'assignment3.ipynb', 'Assignments_DL for NLP.pdf']


In [9]:
import requests
import os

url = "http://d2l-data.s3-accelerate.amazonaws.com/fra-eng.zip"
zip_path = "fra-eng.zip"

print("Downloading English-French dataset...")

response = requests.get(url, timeout=120)

print("Status code:", response.status_code)

response.raise_for_status()

with open(zip_path, "wb") as f:
    f.write(response.content)

print("Download completed!")
print("File size:", os.path.getsize(zip_path), "bytes")

Status code: 200
Download completed!
File size: 3420152 bytes


In [10]:
import zipfile
import os

with zipfile.ZipFile("fra-eng.zip", "r") as zip_ref:
    zip_ref.extractall(".")

print("Dataset extracted successfully!")
print(os.listdir("fra-eng"))

Dataset extracted successfully!
['fra.txt', '_about.txt']


In [11]:
file_path = "fra-eng/fra.txt"

print("Dataset exists:", os.path.exists(file_path))

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Number of sentence pairs:", len(lines))

print("\nFirst 10 examples:")
for line in lines[:10]:
    print(line.strip())

Dataset exists: True
Number of sentence pairs: 167130

First 10 examples:
Go.	Va !
Hi.	Salut !
Run!	Cours !
Run!	Courez !
Who?	Qui ?
Wow!	Ça alors !
Fire!	Au feu !
Help!	À l'aide !
Jump.	Saute.
Stop!	Ça suffit !


In [12]:
import re
import pandas as pd

file_path = "fra-eng/fra.txt"

english_sentences = []
french_sentences = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")

        if len(parts) >= 2:
            english = parts[0]
            french = parts[1]

            english_sentences.append(english)
            french_sentences.append(french)

print("Total English sentences:", len(english_sentences))
print("Total French sentences:", len(french_sentences))

print("\nExample:")
print("English:", english_sentences[0])
print("French :", french_sentences[0])

Total English sentences: 167130
Total French sentences: 167130

Example:
English: Go.
French : Va !


In [13]:
def clean_sentence(sentence):
    sentence = sentence.lower().strip()

    # Keep letters, French accented characters, spaces and basic punctuation
    sentence = re.sub(
        r"[^a-zàâçéèêëîïôûùüÿñæœ'!? ]",
        "",
        sentence
    )

    sentence = re.sub(r"\s+", " ", sentence)

    return sentence


english_sentences = [clean_sentence(s) for s in english_sentences]
french_sentences = [clean_sentence(s) for s in french_sentences]

print("Cleaning completed!")

print("\nExamples after cleaning:")
for i in range(10):
    print("English:", english_sentences[i])
    print("French :", french_sentences[i])
    print()

Cleaning completed!

Examples after cleaning:
English: go
French : va !

English: hi
French : salut !

English: run!
French : cours!

English: run!
French : courez!

English: who?
French : qui ?

English: wow!
French : ça alors!

English: fire!
French : au feu !

English: help!
French : à l'aide!

English: jump
French : saute

English: stop!
French : ça suffit!



In [21]:
french_sentences = [
    "<start> " + sentence + " <end>"
    for sentence in french_sentences
]

print(french_sentences[:10])

['<start> <start> va ! <end> <end>', '<start> <start> salut ! <end> <end>', '<start> <start> cours! <end> <end>', '<start> <start> courez! <end> <end>', '<start> <start> qui ? <end> <end>', '<start> <start> ça alors! <end> <end>', '<start> <start> au feu ! <end> <end>', "<start> <start> à l'aide! <end> <end>", '<start> <start> saute <end> <end>', '<start> <start> ça suffit! <end> <end>']


In [19]:
# Add the required Hello -> Bonjour example
english_data.append("hello")
french_data.append("<start> bonjour ! <end>")

print("English samples:", len(english_data))
print("French samples :", len(french_data))

print("\nLast example:")
print("English:", english_data[-1])
print("French :", french_data[-1])

English samples: 10001
French samples : 10001

Last example:
English: hello
French : <start> bonjour ! <end>


In [22]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Create tokenizers
english_tokenizer = Tokenizer(
    filters='',
    lower=True,
    oov_token='<OOV>'
)

french_tokenizer = Tokenizer(
    filters='',
    lower=True,
    oov_token='<OOV>'
)

# Fit tokenizers on the data
english_tokenizer.fit_on_texts(english_data)
french_tokenizer.fit_on_texts(french_data)

# Vocabulary sizes
english_vocab_size = len(english_tokenizer.word_index) + 1
french_vocab_size = len(french_tokenizer.word_index) + 1

print("English vocabulary size:", english_vocab_size)
print("French vocabulary size :", french_vocab_size)

English vocabulary size: 2566
French vocabulary size : 4859


In [23]:
english_sequences = english_tokenizer.texts_to_sequences(english_data)
french_sequences = french_tokenizer.texts_to_sequences(french_data)

print("English example:")
print(english_data[-1])
print(english_sequences[-1])

print("\nFrench example:")
print(french_data[-1])
print(french_sequences[-1])

English example:
hello
[657]

French example:
<start> bonjour ! <end>
[2, 425, 5, 3]


In [24]:
max_english_length = max(len(seq) for seq in english_sequences)
max_french_length = max(len(seq) for seq in french_sequences)

print("Maximum English sentence length:", max_english_length)
print("Maximum French sentence length :", max_french_length)

Maximum English sentence length: 5
Maximum French sentence length : 12


In [25]:
english_padded = pad_sequences(
    english_sequences,
    maxlen=max_english_length,
    padding='post'
)

french_padded = pad_sequences(
    french_sequences,
    maxlen=max_french_length,
    padding='post'
)

print("English padded shape:", english_padded.shape)
print("French padded shape :", french_padded.shape)

English padded shape: (10001, 5)
French padded shape : (10001, 12)


In [26]:
decoder_input = french_padded[:, :-1]
decoder_target = french_padded[:, 1:]

print("Decoder input shape :", decoder_input.shape)
print("Decoder target shape:", decoder_target.shape)

Decoder input shape : (10001, 11)
Decoder target shape: (10001, 11)


In [27]:
from sklearn.model_selection import train_test_split

(
    encoder_input_train,
    encoder_input_test,
    decoder_input_train,
    decoder_input_test,
    decoder_target_train,
    decoder_target_test
) = train_test_split(
    english_padded,
    decoder_input,
    decoder_target,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(encoder_input_train))
print("Testing samples :", len(encoder_input_test))

Training samples: 8000
Testing samples : 2001


In [28]:
print("English vocabulary size:", english_vocab_size)
print("French vocabulary size :", french_vocab_size)

print("Maximum English length:", max_english_length)
print("Maximum French length :", max_french_length)

print("\nTraining shapes:")
print("Encoder input :", encoder_input_train.shape)
print("Decoder input :", decoder_input_train.shape)
print("Decoder target:", decoder_target_train.shape)

print("\nTesting shapes:")
print("Encoder input :", encoder_input_test.shape)
print("Decoder input :", decoder_input_test.shape)
print("Decoder target:", decoder_target_test.shape)

English vocabulary size: 2566
French vocabulary size : 4859
Maximum English length: 5
Maximum French length : 12

Training shapes:
Encoder input : (8000, 5)
Decoder input : (8000, 11)
Decoder target: (8000, 11)

Testing shapes:
Encoder input : (2001, 5)
Decoder input : (2001, 11)
Decoder target: (2001, 11)


In [29]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

# -----------------------------
# Model parameters
# -----------------------------
EMBEDDING_DIM = 128
LSTM_UNITS = 128

# -----------------------------
# Encoder
# -----------------------------
encoder_inputs = Input(
    shape=(max_english_length,),
    name="encoder_inputs"
)

encoder_embedding = Embedding(
    input_dim=english_vocab_size,
    output_dim=EMBEDDING_DIM,
    name="encoder_embedding"
)(encoder_inputs)

encoder_lstm = LSTM(
    LSTM_UNITS,
    return_state=True,
    name="encoder_lstm"
)

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding
)

# Context vector
encoder_states = [state_h, state_c]


# -----------------------------
# Decoder
# -----------------------------
decoder_inputs = Input(
    shape=(max_french_length - 1,),
    name="decoder_inputs"
)

decoder_embedding_layer = Embedding(
    input_dim=french_vocab_size,
    output_dim=EMBEDDING_DIM,
    name="decoder_embedding"
)

decoder_embedding = decoder_embedding_layer(
    decoder_inputs
)

decoder_lstm = LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)


# -----------------------------
# Dense output layer
# -----------------------------
decoder_dense = Dense(
    french_vocab_size,
    activation="softmax",
    name="output_dense"
)

decoder_outputs = decoder_dense(
    decoder_outputs
)


# -----------------------------
# Complete model
# -----------------------------
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 11)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 5, 128)    │    328,448 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 11, 128)   │    621,952 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │    131,584 │ encoder_embeddin… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 11, 128), │    131,584 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_dense        │ (None, 11, 4859)  │    626,811 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,840,379 (7.02 MB)

 Trainable params: 1,840,379 (7.02 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
encoder_lstm = LSTM(
    LSTM_UNITS,
    return_state=True
)

In [31]:
decoder_lstm = LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True
)

In [32]:
Dense(
    french_vocab_size,
    activation="softmax"
)

<Dense name=dense, built=False>

In [33]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

Model compiled successfully!


In [34]:
sample_output = model.predict(
    [
        encoder_input_train[:2],
        decoder_input_train[:2]
    ],
    verbose=0
)

print("Encoder input shape :", encoder_input_train[:2].shape)
print("Decoder input shape :", decoder_input_train[:2].shape)
print("Model output shape  :", sample_output.shape)

Encoder input shape : (2, 5)
Decoder input shape : (2, 11)
Model output shape  : (2, 11, 4859)


In [ ]:
BATCH_SIZE = 64
EPOCHS = 50

history_128_50 = model.fit(
    [encoder_input_train, decoder_input_train],
    decoder_target_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    shuffle=True
)

Epoch 1/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 8s 53ms/step - accuracy: 0.5939 - loss: 3.5057 - val_accuracy: 0.6215 - val_loss: 2.2758
Epoch 2/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.6910 - loss: 2.1048 - val_accuracy: 0.7051 - val_loss: 2.0396
Epoch 3/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.7090 - loss: 1.9413 - val_accuracy: 0.7105 - val_loss: 1.9470
Epoch 4/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.7140 - loss: 1.8430 - val_accuracy: 0.7119 - val_loss: 1.8738
Epoch 5/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.7161 - loss: 1.7647 - val_accuracy: 0.7141 - val_loss: 1.8231
Epoch 6/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.7229 - loss: 1.6937 - val_accuracy: 0.7274 - val_loss: 1.7604
Epoch 7/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.7326 - loss: 1.6155 - val_accuracy: 0.7359 - val_loss: 1.7030
Epoch 8/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.7464 - loss: 1.5360 - val_accu